# Impact of 401(k) on Financial Wealth

This notebook reproduces the DoubleML example estimating the effect of 401(k) eligibility (`e401`) and participation (`p401`) on net financial assets (`net_tfa`), based on Chernozhukov et al. (2018) and the official [DoubleML 401(k) example](https://docs.doubleml.org/stable/examples/py_double_ml_pension.html).

It is a standalone reference notebook (separate dataset from the scholarship project) used to sanity-check the DoubleML workflow — PLR, IRM, and IIVM — against a well-known, previously-published benchmark result before trusting the same machinery on the scholarship data.

**Identification idea:** 401(k) eligibility is plausibly exogenous once we condition on income and a handful of other observables — around the time 401(k)s became available, employees chose jobs based on income/other job features, not on whether the employer happened to offer a 401(k).

In [ ]:
import numpy as np
import pandas as pd
import doubleml as dml
from doubleml.datasets import fetch_401K

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.pipeline import make_pipeline

from xgboost import XGBClassifier, XGBRegressor

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set()
colors = sns.color_palette()

plt.rcParams['figure.figsize'] = 10., 7.5
sns.set(font_scale=1.5)
sns.set_style('whitegrid', {'axes.spines.top': False,
                             'axes.spines.bottom': False,
                             'axes.spines.left': False,
                             'axes.spines.right': False})

## Data

9,915 household-level observations from the 1991 Survey of Income and Program Participation (SIPP), all variables referenced to 1990.

- `net_tfa` — net financial assets (outcome, $Y$): IRA + 401(k) balances + checking + saving bonds + other interest-earning accounts/assets + stocks + mutual funds, less non-mortgage debt.
- `e401` — eligible for 401(k) (binary)
- `p401` — participates in 401(k) (binary)

An internet connection is required to fetch the dataset.

In [ ]:
data = fetch_401K(return_type='DataFrame')
data.head()

In [ ]:
data.describe()

In [ ]:
data['e401'].value_counts().plot(kind='bar', color=colors)
plt.title('Eligibility, 401(k)')
plt.xlabel('e401')
_ = plt.ylabel('count')

In [ ]:
data['p401'].value_counts().plot(kind='bar', color=colors)
plt.title('Participation, 401(k)')
plt.xlabel('p401')
_ = plt.ylabel('count')

Eligibility is highly associated with financial wealth:

In [ ]:
_ = sns.displot(data, x="net_tfa", hue="e401", col="e401",
                kind="kde", fill=True)

### Naive (unconditional) baseline

These unconditional average predictive effects (APE) do **not** account for saver heterogeneity or the endogeneity of the participation decision, so they are biased — likely overstated — relative to the causal effect.

In [ ]:
print(data[['e401', 'net_tfa']].groupby('e401').mean().diff())

In [ ]:
print(data[['p401', 'net_tfa']].groupby('p401').mean().diff())

## Estimating the ATE of 401(k) Eligibility on Net Financial Assets

Partially linear model:

$$Y = \theta D + g(X) + U, \qquad D = m(X) + V$$

Two covariate specifications:
- **basic**: raw regressors $X$ — used with nonlinear learners (trees, forests, boosting)
- **flexible**: raw regressors + degree-2 orthogonal polynomials of `age`, `inc`, `educ`, `fsize` — used with the linear (lasso) learner

### Data backends

In [ ]:
# Basic model: raw regressors
features_base = ['age', 'inc', 'educ', 'fsize', 'marr',
                  'twoearn', 'db', 'pira', 'hown']

data_dml_base = dml.DoubleMLData(data,
                                  y_col='net_tfa',
                                  d_cols='e401',
                                  x_cols=features_base)
print(data_dml_base)

In [ ]:
# Flexible model: raw regressors + degree-2 polynomials
features = data.copy()[['marr', 'twoearn', 'db', 'pira', 'hown']]

poly_dict = {'age': 2, 'inc': 2, 'educ': 2, 'fsize': 2}
for key, degree in poly_dict.items():
    poly = PolynomialFeatures(degree, include_bias=False)
    data_transf = poly.fit_transform(data[[key]])
    x_cols = poly.get_feature_names_out([key])
    data_transf = pd.DataFrame(data_transf, columns=x_cols)
    features = pd.concat((features, data_transf), axis=1, sort=False)

model_data = pd.concat((data.copy()[['net_tfa', 'e401']], features.copy()),
                        axis=1, sort=False)

data_dml_flex = dml.DoubleMLData(model_data, y_col='net_tfa', d_cols='e401')
print(data_dml_flex)

## Diagnosing Nuisance Function Complexity

DML is only worth the extra machinery if the nuisance functions $g(X) = E[Y \mid X]$ (outcome regression) and $m(X) = E[D \mid X]$ (propensity score) are not actually linear. If they were linear, plain OLS would already partial out $X$ correctly and there'd be no need for cross-fitted ML learners.

Before trusting the lasso/forest/tree/xgboost results above, we check this directly: how much *out-of-sample* predictive power does a linear model give up relative to flexible learners on the exact same nuisance-prediction tasks? A large, consistent gap is evidence the nuisance functions are genuinely nonlinear — which is exactly the regime DML's Neyman-orthogonal score is designed to handle without biasing $\hat\theta$.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression

cv = KFold(n_splits=5, shuffle=True, random_state=42)

X_basic = data[features_base].values
y_arr = data['net_tfa'].values
d_arr = data['e401'].values

outcome_learners = {
    'Linear (OLS)':   LinearRegression(),
    'Random Forest':  RandomForestRegressor(n_estimators=300, max_depth=7,
                                             min_samples_leaf=5, random_state=42, n_jobs=-1),
    'XGBoost':        XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                                    n_jobs=1, objective='reg:squarederror', random_state=42),
}

rows = []
for name, est in outcome_learners.items():
    rmse = -cross_val_score(est, X_basic, y_arr, cv=cv, scoring='neg_root_mean_squared_error')
    r2   = cross_val_score(est, X_basic, y_arr, cv=cv, scoring='r2')
    rows.append({'learner': name, 'CV RMSE': rmse.mean(), 'CV R2': r2.mean()})

outcome_complexity = pd.DataFrame(rows).set_index('learner')
print("Nuisance model g(X) = E[Y | X]  —  5-fold out-of-sample fit by learner")
print(outcome_complexity)

**How to read this table:** all three learners see the same folds, so the comparison is apples-to-apples out-of-sample (not the in-sample $R^2$, which always favors the more flexible model). Look at:
- **CV RMSE** — lower is better. If Random Forest / XGBoost achieve meaningfully lower RMSE than the linear model, the gap *is* the nonlinearity that a linear $g(X)$ cannot capture (e.g. interactions between income and education, or threshold/saturation effects in wealth accumulation).
- **CV R2** — higher is better. A linear $R^2$ well below the ML learners' $R^2$ on the same data is the same signal in normalized form.

If the three rows were nearly identical, that would argue $g(X)$ is close to linear and a plain OLS-partialling-out approach would already be close to efficient.

In [ ]:
from sklearn.metrics import roc_auc_score

propensity_learners = {
    'Logistic (linear)': LogisticRegression(max_iter=1000),
    'Random Forest':     RandomForestClassifier(n_estimators=300, max_depth=5,
                                                 min_samples_leaf=7, random_state=42, n_jobs=-1),
    'XGBoost':           XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                        n_jobs=1, objective='binary:logistic',
                                        eval_metric='logloss', random_state=42),
}

rows = []
for name, est in propensity_learners.items():
    auc = cross_val_score(est, X_basic, d_arr, cv=cv, scoring='roc_auc')
    nll = -cross_val_score(est, X_basic, d_arr, cv=cv, scoring='neg_log_loss')
    rows.append({'learner': name, 'CV AUC': auc.mean(), 'CV LogLoss': nll.mean()})

propensity_complexity = pd.DataFrame(rows).set_index('learner')
print("Nuisance model m(X) = E[D | X] (propensity for e401)  —  5-fold out-of-sample fit by learner")
print(propensity_complexity)

**How to read this table:** AUC of 0.5 means the model is no better than chance at telling who is eligible; AUC of 1.0 is perfect separation.
- **CV AUC** — higher is better. If the ML propensity models clearly outperform logistic regression, the assignment process $m(X)$ has nonlinear/interaction structure (plausible — e.g. eligibility may depend jointly on firm size, income bracket, and pension type in ways a single linear index can't represent).
- **CV LogLoss** — lower is better; penalizes overconfident wrong predictions, so it is more sensitive than AUC to mis-calibration in the tails. A large logistic-vs-ML gap here matters a lot for IRM/IIVM, since both use $\hat m(X)$ as inverse-propensity weights — a poorly specified linear propensity model can produce extreme, unstable weights for individuals it systematically mis-scores.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

rf_outcome = RandomForestRegressor(n_estimators=500, max_depth=7, max_features=3,
                                    min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_outcome.fit(X_basic, y_arr)
lin_outcome = LinearRegression().fit(X_basic, y_arr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat in zip(axes, ['age', 'inc']):
    feat_idx = features_base.index(feat)
    PartialDependenceDisplay.from_estimator(
        rf_outcome, X_basic, [feat_idx], feature_names=features_base, ax=ax,
        line_kw={'label': 'Random Forest', 'color': colors[0]})

    # overlay the linear model's (forced-straight) partial effect for comparison
    grid = np.linspace(data[feat].min(), data[feat].max(), 50)
    X_grid = np.tile(X_basic.mean(axis=0), (50, 1))
    X_grid[:, feat_idx] = grid
    ax.plot(grid, lin_outcome.predict(X_grid), color=colors[1], linestyle='--', label='Linear (OLS)')
    ax.set_title(f'Partial dependence of net_tfa on {feat}')
    ax.legend()
plt.tight_layout()

**How to read this plot:** the dashed line is forced straight by construction — it is the single linear coefficient on that feature, holding the rest of $X$ at its mean. The solid Random Forest curve is free to bend.
- Where the two lines diverge (e.g. a flattening/saturating effect of `inc` at high income, or a hump-shaped effect of `age` around mid-career), that divergence *is* the nonlinearity a linear nuisance model would silently force into a single slope — biasing $\hat g(X)$ for those individuals.
- A roughly straight Random Forest curve that hugs the dashed line would instead support using a linear nuisance model for that variable.

In [ ]:
residuals = y_arr - lin_outcome.predict(X_basic)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(data['inc'], residuals, s=8, alpha=0.3, color=colors[2])
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('inc')
ax.set_ylabel('OLS residual (net_tfa)')
_ = ax.set_title('Linear outcome-model residuals vs. income')

**How to read this plot:** if the linear model were correctly specified, residuals should scatter randomly around zero with no trend across `inc`. A visible trend, fan/funnel shape, or curvature instead means the linear model is systematically under- or over-predicting `net_tfa` at certain income levels — structure a flexible learner could pick up but a straight line cannot. Combined with the CV RMSE/AUC gaps and the PDP curvature above, this is the evidence base for preferring ML-based nuisance learners over plain OLS in this dataset.

## OLS Baseline (Linear, Regression-Adjusted)

To make the value of DML concrete, we add one more comparison point alongside the naive unconditional difference computed earlier: a single linear regression of the outcome on the treatment **and** the controls, $Y = \alpha + \theta D + X\beta + \varepsilon$, fit once by OLS with no cross-fitting and no ML. This is the estimate an analyst would get from `statsmodels`/`lm()` without ever touching DoubleML. We'll carry its coefficient and CI alongside the PLR, IRM, and IIVM results in every comparison table and plot below.

In [ ]:
import statsmodels.api as sm


def ols_baseline(df, y_col, d_col, x_cols, robust=True):
    """Regression-adjusted OLS baseline: Y ~ D + X, no cross-fitting, no ML.

    Returns a one-row DataFrame with the same columns as DoubleML's ``.summary``
    (coef, std err, t, P>|t|, 2.5 %, 97.5 %), indexed by d_col, so it can be
    concatenated directly with PLR/IRM/IIVM summaries for comparison.

    For an endogenous/instrumented target (e.g. p401), this ignores the
    endogeneity and is only a naive contrast point, not a valid LATE estimate.
    """
    X = sm.add_constant(pd.concat([df[[d_col]], df[x_cols]], axis=1))
    y = df[y_col]
    fit = sm.OLS(y, X).fit(cov_type='HC3' if robust else 'nonrobust')
    ci = fit.conf_int().loc[d_col]
    return pd.DataFrame({
        'coef':    [fit.params[d_col]],
        'std err': [fit.bse[d_col]],
        't':       [fit.tvalues[d_col]],
        'P>|t|':   [fit.pvalues[d_col]],
        '2.5 %':   [ci[0]],
        '97.5 %':  [ci[1]],
    }, index=[d_col])

In [ ]:
flex_x_cols = data_dml_flex.x_cols  # raw + degree-2 polynomial features, same spec as lasso

# OLS baseline for the ATE of eligibility (e401) — compared against PLR & IRM below
ols_e401_summary = ols_baseline(model_data, 'net_tfa', 'e401', flex_x_cols)
print("OLS baseline — ATE of e401 (linear, regression-adjusted, flexible covariates)")
print(ols_e401_summary)

# Naive OLS for participation (p401) — NOT instrumented, ignores endogenous self-selection
# into participation. Kept only as a biased contrast point against the IIVM LATE below.
ols_p401_summary = ols_baseline(data, 'net_tfa', 'p401', features_base)
print("\nOLS baseline — naive effect of p401, NOT instrumented (biased; compare to IIVM LATE)")
print(ols_p401_summary)

**How to read these against the DML results below:** `ols_e401_summary` answers the same estimand as the PLR/IRM models (ATE of eligibility) but assumes $g(X)$ and $m(X)$ are linear and uses no cross-fitting — so any difference from the PLR/IRM point estimates is attributable to that linearity assumption plus regularization bias from not cross-fitting. `ols_p401_summary` answers a *different*, biased question (the naive effect of participation ignoring self-selection) and is shown next to the IIVM LATE purely to illustrate how large that selection bias is — it should sit well above the IIVM estimate, similar to how the unconditional p401 difference (≈27,372) sits above everything else.

### Partially Linear Regression Model (PLR)

3-fold cross-fitting throughout. Lasso uses the flexible specification; tree-based learners use the basic specification.

In [ ]:
# Lasso (flexible model)
Cs = 0.0001 * np.logspace(0, 4, 10)
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=10000))
lasso_class = make_pipeline(StandardScaler(),
                             LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear',
                                                   Cs=Cs, max_iter=1000))

np.random.seed(123)
dml_plr_lasso = dml.DoubleMLPLR(data_dml_flex,
                                 ml_l=lasso,
                                 ml_m=lasso_class,
                                 n_folds=3)
dml_plr_lasso.fit(store_predictions=True)
lasso_summary = dml_plr_lasso.summary
print(lasso_summary)

In [ ]:
# Random Forest (basic model)
randomForest = RandomForestRegressor(
    n_estimators=500, max_depth=7, max_features=3, min_samples_leaf=3)
randomForest_class = RandomForestClassifier(
    n_estimators=500, max_depth=5, max_features=4, min_samples_leaf=7)

np.random.seed(123)
dml_plr_forest = dml.DoubleMLPLR(data_dml_base,
                                  ml_l=randomForest,
                                  ml_m=randomForest_class,
                                  n_folds=3)
dml_plr_forest.fit(store_predictions=True)
forest_summary = dml_plr_forest.summary
print(forest_summary)

In [ ]:
# Regression tree (basic model)
trees = DecisionTreeRegressor(
    max_depth=30, ccp_alpha=0.0047, min_samples_split=203, min_samples_leaf=67)
trees_class = DecisionTreeClassifier(
    max_depth=30, ccp_alpha=0.0042, min_samples_split=104, min_samples_leaf=34)

np.random.seed(123)
dml_plr_tree = dml.DoubleMLPLR(data_dml_base,
                                ml_l=trees,
                                ml_m=trees_class,
                                n_folds=3)
dml_plr_tree.fit(store_predictions=True)
tree_summary = dml_plr_tree.summary
print(tree_summary)

In [ ]:
# Boosted trees / XGBoost (basic model)
boost = XGBRegressor(n_jobs=1, objective="reg:squarederror",
                      eta=0.1, n_estimators=35)
boost_class = XGBClassifier(use_label_encoder=False, n_jobs=1,
                             objective="binary:logistic", eval_metric="logloss",
                             eta=0.1, n_estimators=34)

np.random.seed(123)
dml_plr_boost = dml.DoubleMLPLR(data_dml_base,
                                 ml_l=boost,
                                 ml_m=boost_class,
                                 n_folds=3)
dml_plr_boost.fit(store_predictions=True)
boost_summary = dml_plr_boost.summary
print(boost_summary)

In [ ]:
plr_summary = pd.concat((lasso_summary, forest_summary, tree_summary, boost_summary,
                          ols_e401_summary))
plr_summary.index = ['lasso', 'forest', 'tree', 'xgboost', 'ols']
print(plr_summary[['coef', '2.5 %', '97.5 %']])

In [ ]:
errors = np.full((2, plr_summary.shape[0]), np.nan)
errors[0, :] = plr_summary['coef'] - plr_summary['2.5 %']
errors[1, :] = plr_summary['97.5 %'] - plr_summary['coef']
plt.errorbar(plr_summary.index, plr_summary.coef, fmt='o', yerr=errors)
plt.ylim([min(0, plr_summary['2.5 %'].min() * 1.1), plr_summary['97.5 %'].max() * 1.15])
plt.title('Partially Linear Regression Model (PLR) vs. OLS baseline')
plt.xlabel('ML method')
_ = plt.ylabel('Coefficients and 95%-CI')

## Interactive Regression Model (IRM)

Allows for fully heterogeneous treatment effects:

$$Y = g(D, X) + U, \qquad D = m(X) + V$$

Propensity scores near 0/1 are trimmed (`trimming_threshold=0.01`) to limit the influence of extreme weights.

In [ ]:
# Lasso (flexible model)
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=20000))

np.random.seed(123)
dml_irm_lasso = dml.DoubleMLIRM(data_dml_flex,
                                 ml_g=lasso,
                                 ml_m=lasso_class,
                                 trimming_threshold=0.01,
                                 n_folds=3)
dml_irm_lasso.fit(store_predictions=True)
lasso_summary = dml_irm_lasso.summary
print(lasso_summary)

In [ ]:
# Random Forest (basic model) — nuisance-specific hyperparameters
randomForest = RandomForestRegressor(n_estimators=500)
randomForest_class = RandomForestClassifier(n_estimators=500)

np.random.seed(123)
dml_irm_forest = dml.DoubleMLIRM(data_dml_base,
                                  ml_g=randomForest,
                                  ml_m=randomForest_class,
                                  trimming_threshold=0.01,
                                  n_folds=3)

dml_irm_forest.set_ml_nuisance_params('ml_g0', 'e401', {
    'max_depth': 6, 'max_features': 4, 'min_samples_leaf': 7})
dml_irm_forest.set_ml_nuisance_params('ml_g1', 'e401', {
    'max_depth': 6, 'max_features': 3, 'min_samples_leaf': 5})
dml_irm_forest.set_ml_nuisance_params('ml_m', 'e401', {
    'max_depth': 6, 'max_features': 3, 'min_samples_leaf': 6})

dml_irm_forest.fit(store_predictions=True)
forest_summary = dml_irm_forest.summary
print(forest_summary)

In [ ]:
# Regression tree (basic model) — nuisance-specific hyperparameters
trees = DecisionTreeRegressor(max_depth=30)
trees_class = DecisionTreeClassifier(max_depth=30)

np.random.seed(123)
dml_irm_tree = dml.DoubleMLIRM(data_dml_base,
                                ml_g=trees,
                                ml_m=trees_class,
                                trimming_threshold=0.01,
                                n_folds=3)

dml_irm_tree.set_ml_nuisance_params('ml_g0', 'e401', {
    'ccp_alpha': 0.0016, 'min_samples_split': 74, 'min_samples_leaf': 24})
dml_irm_tree.set_ml_nuisance_params('ml_g1', 'e401', {
    'ccp_alpha': 0.0018, 'min_samples_split': 70, 'min_samples_leaf': 23})
dml_irm_tree.set_ml_nuisance_params('ml_m', 'e401', {
    'ccp_alpha': 0.0028, 'min_samples_split': 167, 'min_samples_leaf': 55})

dml_irm_tree.fit(store_predictions=True)
tree_summary = dml_irm_tree.summary
print(tree_summary)

In [ ]:
# Boosted trees / XGBoost (basic model) — nuisance-specific hyperparameters
boost = XGBRegressor(n_jobs=1, objective="reg:squarederror")
boost_class = XGBClassifier(use_label_encoder=False, n_jobs=1,
                             objective="binary:logistic", eval_metric="logloss")

np.random.seed(123)
dml_irm_boost = dml.DoubleMLIRM(data_dml_base,
                                 ml_g=boost,
                                 ml_m=boost_class,
                                 trimming_threshold=0.01,
                                 n_folds=3)

dml_irm_boost.set_ml_nuisance_params('ml_g0', 'e401', {
    'eta': 0.1, 'n_estimators': 8})
dml_irm_boost.set_ml_nuisance_params('ml_g1', 'e401', {
    'eta': 0.1, 'n_estimators': 29})
dml_irm_boost.set_ml_nuisance_params('ml_m', 'e401', {
    'eta': 0.1, 'n_estimators': 23})

dml_irm_boost.fit(store_predictions=True)
boost_summary = dml_irm_boost.summary
print(boost_summary)

In [ ]:
irm_summary = pd.concat((lasso_summary, forest_summary, tree_summary, boost_summary,
                          ols_e401_summary))
irm_summary.index = ['lasso', 'forest', 'tree', 'xgboost', 'ols']
print(irm_summary[['coef', '2.5 %', '97.5 %']])

In [ ]:
errors = np.full((2, irm_summary.shape[0]), np.nan)
errors[0, :] = irm_summary['coef'] - irm_summary['2.5 %']
errors[1, :] = irm_summary['97.5 %'] - irm_summary['coef']
plt.errorbar(irm_summary.index, irm_summary.coef, fmt='o', yerr=errors)
plt.ylim([min(0, irm_summary['2.5 %'].min() * 1.1), irm_summary['97.5 %'].max() * 1.15])
plt.title('Interactive Regression Model (IRM) vs. OLS baseline')
plt.xlabel('ML method')
_ = plt.ylabel('Coefficients and 95%-CI')

These confounder-adjusted PLR/IRM estimates are substantially attenuated relative to the naive unconditional baseline (~19,559), suggesting the naive estimate is upward-biased by saver heterogeneity.

## Local Average Treatment Effect of 401(k) Participation — Interactive IV Model (IIVM)

Now we estimate the LATE of *participation* (`p401`), using *eligibility* (`e401`) as a binary instrument for the endogenous participation decision. This identifies the effect for compliers — individuals who participate only because they are eligible.

Structural model:

$$Y = g(D, X) + U, \qquad D = m(Z, X) + V, \qquad Z = e401 \text{ (instrument)}$$

In [ ]:
# Basic model with instrument
data_dml_base_iv = dml.DoubleMLData(data,
                                     y_col='net_tfa',
                                     d_cols='p401',
                                     z_cols='e401',
                                     x_cols=features_base)
print(data_dml_base_iv)

In [ ]:
# Flexible model with instrument
model_data = pd.concat((data.copy()[['net_tfa', 'e401', 'p401']], features.copy()),
                        axis=1, sort=False)

data_dml_iv_flex = dml.DoubleMLData(model_data,
                                     y_col='net_tfa',
                                     d_cols='p401',
                                     z_cols='e401')
print(data_dml_iv_flex)

In [ ]:
# Lasso (flexible model)
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=20000))

np.random.seed(123)
dml_iivm_lasso = dml.DoubleMLIIVM(data_dml_iv_flex,
                                   ml_g=lasso,
                                   ml_m=lasso_class,
                                   ml_r=lasso_class,
                                   subgroups={'always_takers': False,
                                              'never_takers': True},
                                   trimming_threshold=0.01,
                                   n_folds=3)
dml_iivm_lasso.fit(store_predictions=True)
lasso_summary = dml_iivm_lasso.summary
print(lasso_summary)

In [ ]:
# Random Forest (basic model)
randomForest = RandomForestRegressor(n_estimators=500)
randomForest_class = RandomForestClassifier(n_estimators=500)

np.random.seed(123)
dml_iivm_forest = dml.DoubleMLIIVM(data_dml_base_iv,
                                    ml_g=randomForest,
                                    ml_m=randomForest_class,
                                    ml_r=randomForest_class,
                                    subgroups={'always_takers': False,
                                               'never_takers': True},
                                    trimming_threshold=0.01,
                                    n_folds=3)

dml_iivm_forest.set_ml_nuisance_params('ml_g0', 'p401', {
    'max_depth': 6, 'max_features': 4, 'min_samples_leaf': 7})
dml_iivm_forest.set_ml_nuisance_params('ml_g1', 'p401', {
    'max_depth': 6, 'max_features': 3, 'min_samples_leaf': 5})
dml_iivm_forest.set_ml_nuisance_params('ml_m', 'p401', {
    'max_depth': 6, 'max_features': 3, 'min_samples_leaf': 6})
dml_iivm_forest.set_ml_nuisance_params('ml_r1', 'p401', {
    'max_depth': 4, 'max_features': 7, 'min_samples_leaf': 6})

dml_iivm_forest.fit(store_predictions=True)
forest_summary = dml_iivm_forest.summary
print(forest_summary)

In [ ]:
# Regression tree (basic model)
trees = DecisionTreeRegressor(max_depth=30)
trees_class = DecisionTreeClassifier(max_depth=30)

np.random.seed(123)
dml_iivm_tree = dml.DoubleMLIIVM(data_dml_base_iv,
                                  ml_g=trees,
                                  ml_m=trees_class,
                                  ml_r=trees_class,
                                  subgroups={'always_takers': False,
                                             'never_takers': True},
                                  trimming_threshold=0.01,
                                  n_folds=3)

dml_iivm_tree.set_ml_nuisance_params('ml_g0', 'p401', {
    'ccp_alpha': 0.0016, 'min_samples_split': 74, 'min_samples_leaf': 24})
dml_iivm_tree.set_ml_nuisance_params('ml_g1', 'p401', {
    'ccp_alpha': 0.0018, 'min_samples_split': 70, 'min_samples_leaf': 23})
dml_iivm_tree.set_ml_nuisance_params('ml_m', 'p401', {
    'ccp_alpha': 0.0028, 'min_samples_split': 167, 'min_samples_leaf': 55})
dml_iivm_tree.set_ml_nuisance_params('ml_r1', 'p401', {
    'ccp_alpha': 0.0576, 'min_samples_split': 55, 'min_samples_leaf': 18})

dml_iivm_tree.fit(store_predictions=True)
tree_summary = dml_iivm_tree.summary
print(tree_summary)

In [ ]:
# Boosted trees / XGBoost (basic model)
boost = XGBRegressor(n_jobs=1, objective="reg:squarederror")
boost_class = XGBClassifier(use_label_encoder=False, n_jobs=1,
                             objective="binary:logistic", eval_metric="logloss")

np.random.seed(123)
dml_iivm_boost = dml.DoubleMLIIVM(data_dml_base_iv,
                                   ml_g=boost,
                                   ml_m=boost_class,
                                   ml_r=boost_class,
                                   subgroups={'always_takers': False,
                                              'never_takers': True},
                                   trimming_threshold=0.01,
                                   n_folds=3)

dml_iivm_boost.set_ml_nuisance_params('ml_g0', 'p401', {
    'eta': 0.1, 'n_estimators': 9})
dml_iivm_boost.set_ml_nuisance_params('ml_g1', 'p401', {
    'eta': 0.1, 'n_estimators': 33})
dml_iivm_boost.set_ml_nuisance_params('ml_m', 'p401', {
    'eta': 0.1, 'n_estimators': 12})
dml_iivm_boost.set_ml_nuisance_params('ml_r1', 'p401', {
    'eta': 0.1, 'n_estimators': 25})

dml_iivm_boost.fit(store_predictions=True)
boost_summary = dml_iivm_boost.summary
print(boost_summary)

In [ ]:
iivm_summary = pd.concat((lasso_summary, forest_summary, tree_summary, boost_summary,
                           ols_p401_summary))
iivm_summary.index = ['lasso', 'forest', 'tree', 'xgboost', 'ols (naive, biased)']
print(iivm_summary[['coef', '2.5 %', '97.5 %']])

In [ ]:
colors = sns.color_palette()

errors = np.full((2, iivm_summary.shape[0]), np.nan)
errors[0, :] = iivm_summary['coef'] - iivm_summary['2.5 %']
errors[1, :] = iivm_summary['97.5 %'] - iivm_summary['coef']
plt.errorbar(iivm_summary.index, iivm_summary.coef, fmt='o', yerr=errors)
plt.ylim([min(0, iivm_summary['2.5 %'].min() * 1.1), iivm_summary['97.5 %'].max() * 1.15])
plt.title('Interactive IV Model (IIVM) vs. naive OLS baseline')
plt.xlabel('ML method')
_ = plt.ylabel('Coefficients and 95%-CI')

## Summary of Results

Combine PLR (ATE of eligibility), IRM (ATE of eligibility, heterogeneous), and IIVM (LATE of participation, instrumented by eligibility) across all four ML methods.

In [ ]:
df_summary = pd.concat((plr_summary, irm_summary, iivm_summary)).reset_index().rename(columns={'index': 'ML'})
df_summary['Model'] = np.concatenate((np.repeat('PLR', len(plr_summary)),
                                       np.repeat('IRM', len(irm_summary)),
                                       np.repeat('IIVM', len(iivm_summary))))
df_summary.set_index(['Model', 'ML'])

In [ ]:
plt.figure(figsize=(10, 15))
colors = sns.color_palette()
for ind, model in enumerate(['PLR', 'IRM', 'IIVM']):
    plt.subplot(3, 1, ind + 1)
    this_df = df_summary.query('Model == @model')
    errors = np.full((2, this_df.shape[0]), np.nan)
    errors[0, :] = this_df['coef'] - this_df['2.5 %']
    errors[1, :] = this_df['97.5 %'] - this_df['coef']
    plt.errorbar(this_df.ML, this_df.coef, fmt='o', yerr=errors,
                 color=colors[ind], ecolor=colors[ind])
    plt.ylim([min(0, this_df['2.5 %'].min() * 1.1), this_df['97.5 %'].max() * 1.15])
    plt.title(model)
    plt.ylabel('Coefficients and 95%-CI')

_ = plt.xlabel('ML method')

Note the `ols` rows: for PLR/IRM they answer the same ATE-of-eligibility question under a linearity assumption, so a close match to the ML rows is reassurance, not redundancy. The `ols (naive, biased)` row in the IIVM panel answers a different, uninstrumented question — its distance from the IIVM LATE estimates is a direct visual measure of the participation self-selection bias that the instrument is correcting for.

**Takeaway:** treatment-effect estimates are stable across the four ML methods within each model class, and all are highly statistically significant — we would reject the hypothesis that 401(k) participation has no effect on financial wealth. The DML-adjusted PLR/IRM estimates of the eligibility effect (~8,000–9,000) are far below the naive unconditional estimate (~19,559), and the IIVM LATE of participation (~11,000–13,000) is correspondingly larger than the eligibility ATE, consistent with eligibility being a (noisy) instrument for the more potent participation decision.